# 4.1.5 Backpropagation

Chain rule derivation, numpy two-layer MLP with full forward + backward pass on make_moons.

In [ ]:
import numpy as np
from sklearn.datasets import make_moons
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sigmoid=lambda x:1/(1+np.exp(-np.clip(x,-500,500)))
relu=lambda x:np.maximum(0,x); relu_d=lambda x:(x>0).astype(float)

class Net:
    def __init__(self,ni,nh,lr=.05,seed=42):
        rng=np.random.default_rng(seed); self.lr=lr
        self.W1=rng.standard_normal((ni,nh))*np.sqrt(2/ni); self.b1=np.zeros(nh)
        self.W2=rng.standard_normal((nh,1))*np.sqrt(2/nh); self.b2=np.zeros(1)
    def fwd(self,X):
        self.X=X; self.z1=X@self.W1+self.b1; self.a1=relu(self.z1)
        self.z2=self.a1@self.W2+self.b2; self.a2=sigmoid(self.z2); return self.a2
    def bwd(self,y):
        n=len(y); y=y.reshape(-1,1)
        dz2=(self.a2-y)/n; dW2=self.a1.T@dz2; db2=dz2.sum(0)
        da1=dz2@self.W2.T; dz1=da1*relu_d(self.z1); dW1=self.X.T@dz1; db1=dz1.sum(0)
        self.W2-=self.lr*dW2; self.b2-=self.lr*db2; self.W1-=self.lr*dW1; self.b1-=self.lr*db1
    def loss(self,y):
        y=y.reshape(-1,1); p=np.clip(self.a2,1e-7,1-1e-7)
        return -np.mean(y*np.log(p)+(1-y)*np.log(1-p))

X,y=make_moons(800,noise=.2,random_state=42); Xs=StandardScaler().fit_transform(X)
Xt,Xv,yt,yv=train_test_split(Xs,y,test_size=.2,random_state=42)
net=Net(2,32)

In [ ]:
losses=[]
for ep in range(300):
    net.fwd(Xt); net.bwd(yt)
    if ep%50==0: l=net.loss(yt); losses.append(l); print(f'Ep{ep:3d}: loss={l:.4f}  acc={(net.a2[:,0]>.5).astype(int).mean():.4f}')

In [ ]:
import matplotlib.pyplot as plt
plt.plot(range(0,300,50),losses,marker='o'); plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Training Loss'); plt.show()
net.fwd(Xv); print('Test acc:',((net.a2[:,0]>.5).astype(int)==yv).mean())